# MTT 与 TESLA

我们迫切地需要一些优化技术来减少我们在数据集蒸馏时的计算负担，TESLA 试图解决这个问题。

更多的，MTT 也是一种目前主流的轨迹蒸馏方法。我们先说说 MTT。

# Matching Training Trajectories

推荐你读 https://arxiv.org/pdf/2203.11932 Dataset Distillation by Matching Training Trajectories 这是 MTT 原文。

我想把一件事放在最开头说，那就是 MTT 可能会让人失望。原因是它捡回了 Gradient Matching 抛弃的 Parameter Matching 思想，并且做出了一个类似对于 Curriculum Paramter Matching 的优化的成果。这意味着它具备我们上一章中提到的 PM 多数缺点。

从法理上，首先参数匹配仅仅是模型输出相似的充分而不必要条件。其次的，如果我们考虑整个 Trajectory，我们无可避免地需要对整个轨迹穿透反向传播，这会带来类似原始数据集蒸馏的计算问题。

但是仍有一个好消息，MTT 解决了长程匹配的跨度太大问题。我们详细说。

## 基本逻辑

MTT 的核心思想是用专家轨迹来对齐合成数据集上的轨迹。首先在真实数据集 $\mathcal D_{\mathrm{real}}$ 上训练多个专家轨迹
$$\tau^*
=
\{\theta_t^*\}_{t=0}^{T}$$
对于多个不同初始化专家轨迹，他们是
$$\{\tau_i^*\}$$
这些专家轨迹可以离线预计算，后续所有蒸馏实验共用。

对于每一次蒸馏步骤，我们随机选出一条轨迹 $\tau^*$ 与随机起始时刻 $t$，那么我们可以从专家轨迹中选取参数 
$$\theta_t^*$$

我们将学生参数初始化为此参数 $\hat{\theta}_t=\theta_t^*$。

学生模型更新仅仅使用合成数据集 $\mathcal D_{\mathrm{syn}}$。对于第 $n$ 步更新，其可以写成
$$\hat{\theta}_{t+n+1}
=
\hat{\theta}_{t+n}
-
\alpha
\nabla_\theta
\ell
\left(
\mathcal A(b_{t+n});
\hat{\theta}_{t+n}
\right)$$
其中 $b_{t+n}\sim\mathcal D_{\mathrm{syn}}$ 是从合成数据集取出的 mini-batch，$\mathcal A$ 是我们上一章提到的数据增强算子，$\alpha$ 是学习率。注意，$\alpha$ 也是需要更新的，我们会将其应用在真正的蒸馏数据集学习之中。

所以经历 $N$ 步之后，学生模型抵达
$$\hat{\theta}_{t+N}$$

对于专家轨迹，其从 $\theta_t^*$ 出发进行 $M$ 次更新达到
$$\theta_{t+M}^*$$
我们希望这样一件事
$$\hat{\theta}_{t+N}
\approx
\theta_{t+M}^*$$
这里 $N\ll M$。这个要求乍一看很奇怪但是其实非常合理，我们希望少量合成训练步骤模拟更多真实训练步骤。

最终损失形式是
$$\mathcal L
=
\frac{
\left\|
\hat{\theta}_{t+N}
-
\theta_{t+M}^*
\right\|_2^2
}{
\left\|
\theta_t^*
-
\theta_{t+M}^*
\right\|_2^2
}$$
归一化是必要的，原因是专家轨迹在不同阶段移动距离完全不同。如果我们不用归一化，早期轨迹段可能因为绝对距离大而主导损失。我们希望衡量学生模型相较专家轨迹更新的相对误差。

请注意一下计算图。学生模型最终参数依赖每一步合成数据更新
$$\mathcal D_{\mathrm{syn}}
\longrightarrow
\hat{\theta}_{t+1}
\longrightarrow
\hat{\theta}_{t+2}
\longrightarrow
\cdots
\longrightarrow
\hat{\theta}_{t+N}
\longrightarrow
\mathcal L$$
这意味着对于合成数据集的更新需要穿透 $N$ 个更新步骤，计算 $\nabla_{\mathcal D_{\mathrm{syn}}}\mathcal L$ 并不容易。实际上学习率 $\alpha$ 的更新也大致如此。

还有一件事，那就是我们训练模型时取出的是合成数据集的一个 mini-batch $b_{t+n}\sim \mathcal D_{\mathrm{syn}}$ 而不是完整的合成数据集 $\mathcal{D}_{syn}$。这个技巧可以有效地节省显存，我们尽力减少计算上的负担。

这个技巧也带来一点需要讨论：合成数据集中哪些部分会被更新？实际上如果在一次更新当中某个数据从未被抽取，那么我们更新时对其的梯度为零，换言之我们只更新被抽到的合成数据集中部分。所以为了更合理的更新，我们在实际抽取 mini-batch 时不会完全随机，而是通过某种方法保证每个数据点至少被抽到一次，比如切割分块。

## 蒸馏算法

我们已经基本说过一遍蒸馏算法流程了。下面给出完整算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{Dataset Distillation via Trajectory Matching} \\
\hline
\textbf{Input: } \{\tau_i^*\}\text{: set of expert parameter trajectories trained on } \mathcal{D}_{\text{real}}. \\
\textbf{Input: } M\text{: \# of updates between starting and target expert params.} \\
\textbf{Input: } N\text{: \# of updates to student network per distillation step.} \\
\textbf{Input: } \mathcal{A}\text{: Differentiable augmentation function.} \\
\textbf{Input: } T^+ < T\text{: Maximum start epoch.} \\
\begin{aligned}
1: & \ \text{Initialize distilled data } \mathcal{D}_{\text{syn}} \sim \mathcal{D}_{\text{real}} \\
2: & \ \text{Initialize trainable learning rate } \alpha := \alpha_0 \text{ for apply } \mathcal{D}_{\text{syn}} \\
3: & \ \textbf{for each } \text{distillation step\dots} \textbf{ do} \\
4: & \ \quad \triangleright \text{Sample expert trajectory: } \tau^* \sim \{\tau_i^*\} \text{ with } \tau^* = \{\theta_t^*\}_0^T \\
5: & \ \quad \triangleright \text{Choose random start epoch, } t \le T^+ \\
6: & \ \quad \triangleright \text{Initialize student network with expert params:} \\
7: & \ \quad\quad \hat{\theta}_t := \theta_t^* \\
8: & \ \quad \textbf{for } n = 0 \to N - 1 \textbf{ do} \\
9: & \ \quad\quad \triangleright \text{Sample a mini-batch of distilled images:} \\
10:& \ \quad\quad\quad b_{t+n} \sim \mathcal{D}_{\text{syn}} \\
11:& \ \quad\quad \triangleright \text{Update student network w.r.t. classification loss:} \\
12:& \ \quad\quad\quad \hat{\theta}_{t+n+1} = \hat{\theta}_{t+n} - \alpha \nabla \ell(\mathcal{A}(b_{t+n}); \hat{\theta}_{t+n}) \\
13:& \ \quad \textbf{end for} \\
14:& \ \quad \triangleright \text{Compute loss between ending student and expert params:} \\
15:& \ \quad\quad \mathcal{L} = \|\hat{\theta}_{t+N} - \theta_{t+M}^*\|_2^2 \ / \ \|\theta_t^* - \theta_{t+M}^*\|_2^2 \\
16:& \ \quad \triangleright \text{Update } \mathcal{D}_{\text{syn}} \text{ and } \alpha \text{ with respect to } \mathcal{L} \\
17:& \ \textbf{end for}
\end{aligned} \\
\textbf{Output: } \text{distilled data } \mathcal{D}_{\text{syn}} \text{ and learning rate } \alpha \\
\hline
\end{array}$$

我们提醒一件事，就是实际 Pytorch 实现中，我们无需显式考虑 HVP 优化，因为 Pytorch 自动微分已经为我们做好了这一部分。Pytorch 自动微分底层原理是对于每个计算节点计算 JVP，而 HVP 实际上就是一阶微分被视为向量函数下的 JVP。

从 MTT 开始，这种反向传播的优化开始成为默认，而不是在原始数据集蒸馏那样强调计算的可行性。

## MTT 的总结

MTT 表现出一定的架构迁移性。具体而言，对于 ConvNet 蒸馏的合成数据集对于 ResNet 和 VGG 等等网络也有效果，这表明其没有很严重地过拟合。

更多的，当我们将合成数据集从每类一张扩展到每类十张，模型准确率大幅提升了。但是继续增加合成数据集大小则会产生边际效应。

两个重要的超参数是 $N,M$。作者发现，当这两个值设置得极小，这会退化成一个局部监督，效果不好。当这两个值设置得更大，模型拥有更强的长程拟合能力。前者被称为短程匹配，后者被称为长程匹配。但是也不是越长程越好，相比纯局部匹配，适当的有限长程匹配更强。但直接跨越整个训练过程也可能难以优化。

MTT 声称自己是首个做到 $128 \times 128$ 分辨率 ImageNet 数据集子集蒸馏的方案。这里有一件事很有意思，那就是 MTT 方法计算量对于图像维度不是很敏感，这可能是它能够成为首个方法的原因之一。但是这并不是说 GM 等等方法无法做到，实际上这只是，实验上没人做过。

所以，无论什么方法，我们都极其关心其计算负担。对于计算负担的削减扩展到更大数据集上时也会带来更大的便利。

为你介绍 TESLA，这是一种我们所需要的方法。

# TESLA

推荐你读 https://arxiv.org/pdf/2211.10586 Scaling Up Dataset Distillation to ImageNet-1K with Constant Memory 这是原文。

尽管达到了当时的 State-of-the-Art 性能，MTT 由于其巨大的 GPU 显存需求难以扩展到更大数据集上。原因是 MTT 需要对梯度下降更新参数过程中的每一步计算图保存，这会导致显存需求随着更新步数呈线性增长。

TESLA 提出一种惊人的优化：我们可以通过重排计算图，使得显存由 $O(N)$ 变为 $O(1)$，也就是常数级别显存。

## 基本逻辑

首先回顾一下 MTT。核心损失是
$$\mathcal L
=
\frac{
\left\|
\hat\theta_{t+T}
-
\theta_{t+M}^{*}
\right\|_2^2
}{
\left\|
\theta_t^{*}
-
\theta_{t+M}^{*}
\right\|_2^2
}$$
其中 $\theta_t^*$ 指的是专家轨迹在时刻 $t$ 时的权重，$\theta_{t+M}^*$ 则是专家轨迹再向前训练 $M$ 时刻的权重，$\hat\theta_{t+T}$ 是学生模型从 $\theta_t^*$ 开始在合成数据集上向前训练 $T$ 时刻的权重。

学生的第 $i$ 步更新是
$$\hat\theta_{t+i+1}
=
\hat\theta_{t+i}
-
\beta
\nabla_\theta
\ell\left(
\hat\theta_{t+i};
\widetilde X_i
\right)$$
其中 $\widetilde X_i$ 是第 $i$ 步使用的合成数据集的 mini-batch。

因此我们可以将学生模型在合成数据集上的 $T$ 步更新全部展开
$$\hat\theta_{t+T}
=
\theta_t^*
-
\beta
\sum_{i=0}^{T-1}
\nabla_\theta
\ell\left(
\hat\theta_{t+i};
\widetilde X_i
\right)$$
为了从最终损失反传到每个合成 mini-batch，原始 MTT 会在前向更新时为每一步都保留网络参数梯度的计算图与高阶微分所需的中间变量。

因此驻留的显存是
$$\mathcal G_0,\mathcal G_1,\ldots,\mathcal G_{T-1}$$
其中 $\mathcal G_i$ 表示第 $i$ 步梯度对应的计算图。

如果一个 mini-batch 计算图大约占据显存 $C_{\mathrm{graph}}$ 那么总体就需要占据
$$O\left(
T C_{\mathrm{graph}}
\right)$$
这是一个随着步数 $T$ 线性递增的量级。

现在我们开始指出一些数学上的事实，这些事实可以关键性地揭示，我们只需要保存常数量级的显存。

简化一下符号，我们记
$$g_i
=
\nabla_\theta
\ell\left(
\hat\theta_{t+i};
\widetilde X_i
\right)$$
并且定义
$$G
=
\sum_{i=0}^{T-1}g_i$$
再记
$$\Delta
=
\theta_{t+M}^{*}
-
\theta_t^{*}$$
所以我们可以推出学生模型终点
$$\hat\theta_{t+T}
=
\theta_t^{*}
-
\beta G$$
我们想要的学生模型和专家轨迹之间差距是
$$\hat\theta_{t+T}
-
\theta_{t+M}^{*}
=
-
\left(
\Delta+\beta G
\right)$$
因此
$$\left\|
\hat\theta_{t+T}
-
\theta_{t+M}^{*}
\right\|_2^2
=
\left\|
\Delta+\beta G
\right\|_2^2$$
我们将其展开
$$\left\|
\Delta+\beta G
\right\|_2^2
=
\|\Delta\|_2^2
+
2\beta\Delta^\top G
+
\beta^2\|G\|_2^2 \quad (*)$$

我们重点观察 $(*)$ 式，因为这就是我们需要优化的损失函数原始形式。其中
$$\|\Delta\|_2^2$$
是一个常数，因此我们梯度下降优化时不会关心。

第二项
$$2\beta\Delta^\top G
=
2\beta
\sum_{i=0}^{T-1}
\Delta^\top g_i$$
已经按照 mini-batch 做了分解，每一项涉及的 $g_i$ 非常清晰。

第三项
$$\beta^2
\left\|
\sum_i g_i
\right\|_2^2$$
仅仅关心梯度之和 $G$，这意味着保存 $g_0,\ldots,g_{T-1}$ 并不严格必要。

现在我们开始指出这些数学事实如何被我们利用。我们定义
$$J_i
=
\frac{\partial g_i}
{\partial\widetilde X_i}$$
这是损失梯度关于 mini-batch 的梯度。

我们对 $(*)$ 式子求对于 mini-batch $\widetilde{X}_i$ 的偏导数
$$\nabla_{\widetilde X_i}
\left\|
\hat\theta_{t+T}
-
\theta_{t+M}^{*}
\right\|_2^2
=
2\beta J_i^\top\Delta
+
2\beta^2J_i^\top G = 
2\beta
J_i^\top
\left(
\Delta+\beta G
\right)$$

所以实际上我们并不关系完整 $J_i$ 而是关心一个类似 JVP 形式
$$J_i^\top
\left(
\Delta+\beta G
\right)$$

这意味着当我们尝试获取损失对于第 $i$ 个 mini-batch 的梯度，我们仅仅需要两件事：mini-batch 对于梯度损失 $g_i$ 的偏导数，以及可以视为常量的 $\Delta+\beta G$。

前者仅仅需要权重 $\hat{\theta}_{t+i}$ 对于 mini-batch $\widetilde{X}_i$ 上损失关于权重 $\hat{\theta}_{t+i}$ 的梯度关于 mini-batch $\widetilde{X}_i$ 的梯度，这是一个局部的计算图，仅仅与 $T=i$ 发生的事件相关。而后者是一个完全无关梯度的常量，这意味着我们可以在完全不构建计算图的情况下计算这个量。

我们详细说说这里的算法。

## 优化算法

我们需要进行两遍循环。

首先从 $\theta_t^*$ 开始按照我们所说的学生模型更新方式进行更新，但是我们完全不构建任何计算图
$$\hat\theta_{t+i+1}
=
\hat\theta_{t+i}
-
\beta
\nabla_\theta
\ell\left(
\hat\theta_{t+i};
\widetilde X_i
\right)$$
同时，我们保存模型权重 $\hat\theta_{t+i}$ 与梯度累计和
$$g_i
=
\nabla_{\theta}
\ell(\hat\theta_{t+i};\widetilde X_i), \quad G
\leftarrow
G+g_i.$$

经历第一个循环之后，我们拥有每个时间步下权重 $\hat\theta_{t+i}, i=0,...,T-1$ 与我们所需的常量 $\Delta+\beta G$。请注意这里的 $\Delta$ 甚至不需要在循环中计算而是在生成专家轨迹时进行预计算。

一个注意点是，我们拥有这些权重不意味着需要全部进入显存，甚至是相反的。我们可以在每次计算结束 $\hat\theta_{t+i+1}$ 之后直接将 $\hat\theta_{t+i}$ 存入硬盘储存，这意味着进行第一个循环中计算时显存保持了常数级别 $O(1)$。

现在我们进入第二个循环。

对于第 $i$ 个 mini-batch，首先重新计算
$$g_i
=
\nabla_{\theta}
\ell(\hat\theta_{t+i};\widetilde X_i)$$
这里需要用到我们之前保存的权重 $\hat\theta_{t+i}$。这次计算我们需要保存对于 $g_i$ 的计算图以准备反向传播。

现在我们可以通过一个局部计算图隐式给出 $J_i$
$$\widetilde X_i
\longrightarrow
\ell(\hat\theta_{t+i};\widetilde X_i)
\longrightarrow
g_i.$$
请注意我这里说的是隐式计算，因为实际上我们会直接计算 JVP 而不是显式计算 $J_i$，后者非常庞大
$$2\beta
J_i^\top
\left(
\Delta+\beta G
\right)$$

至此，最终损失对于第 $i$ 个 mini-batch $\widetilde X_i$ 的梯度已经得到。我们将这个梯度存入硬盘储存或者内存，并且直接释放此次计算所用到的所有计算图。这意味着我们计算任意一个 mini-batch $X_i$ 对于最终损失的梯度时，显存中仅仅储存了常数 $O(1)$ 量级的内容。

我们对比一下 MTT。MTT 显存是
$$\operatorname{Memory}_{\mathrm{MTT}}
=
O\left(
T|\widetilde X_i|\mathcal G
\right)$$
TESLA 仅仅是
$$\operatorname{Memory}_{\mathrm{TESLA}}
=
O\left(
|\widetilde X_i|\mathcal G
\right)$$
所以我们会发现 TESLA 做了一件巨大的进步。

更多的，我们检查一件事，假如某个数据点在多个 mini-bacth 中出现，梯度计算是否会出现问题？

实际上不会，这只是简单的梯度累计。我们记 $\pi(i,b)=j$ 表示第 $i$ 个 mini-batch 第 $b$ 个位置出现了 $x_j$，那么对于这个位置的最终梯度实际上就是梯度之和
$$\nabla_{x_j}\mathcal L
=
\sum_{(i,b):\pi(i,b)=j}
\nabla_{\widetilde X_{i,b}}\mathcal L$$

### Soft Label

我们现在来说 TESLA 的第二个贡献，被称为 Soft Label。传统的认知中，被用于训练的数据图片必须拥有一个强硬的单一的分类，因为一张狗的图片毫无疑问归于狗的类别下。

但是我们指出实际上并不是这样。当我们观察交叉熵损失允许的形式，我们会发现实际上一张图片的 Label 可以是更接近概率性质的。换言之，如果一张图片本身就非常像狗和猫，我们不一定非要给他一个狗的 one-hot 标签，例如 (1,0)，或者猫的标签，例如 (0,1)，而是一个狗和猫之间的标签 ，例如 (0.5,0.5)。这就是 Soft Label 的基本思想。

Soft Label 的思想对于数据集蒸馏变得尤为重要，因为合成数据集样本非常稀少，这意味着我们迫切地需要更丰富的真实监督方向。

TESLA 提出了一种 Training-free 的 Soft Label 方法。

首先我们选择专家轨迹的起点 $\theta_t^*$ 和终点 $\theta_{t+M}^*$，然后将我们准备输入学生模型进行训练的合成数据集中图像 $\widetilde x$ 先交给终点专家模型，终点专家模型会输出一个 logits
$$z=
f_{\theta_{t+M}^*}(\widetilde x),$$

现在进行 softmax 得到 Soft Label
$$q_{c}
=
\frac{\exp(z_{c})}
{\sum_{k=1}^{C}\exp(z_{k})}.$$

同样的，模型最终也会输出一个 softmax 之后的归一化 logits，于是我们可以计算交叉熵损失
$$\ell
=
-
\sum_{c=1}^{C}
q_{c}
\log p_{c},$$
这个损失会成为 $\ell\left(
\hat\theta_{t+i};
\widetilde X_i
\right)$ 的一部分。之后的事情与我们在优化算法中所说的一样。

实际上我们并不会对于选出的 mini-batch 在线计算这个 Soft Label 而是直接对于整个合成数据集使用专家 $f_{\theta_{t+M}^*}$ 先做处理。

这里有一点需要讨论，为什么我们选用 $f_{\theta_{t+M}^*}$ 而不是其他专家轨迹节点？实际上这是因为 $\theta_{t+M}^*$ 正是学生模型所追求的权重参数。作者尝试了最后一个专家节点，也就是训练最完好的节点，效果并不好。因为最后的专家节点实际上并不是我们所需要的专家节点。

下面这张图详细展示了 TESLA 的整个蒸馏算法。

<img src="./assets/TESLA.png" width="900" height="300">

下面是完整的数据集蒸馏算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{Trajectory matching with Soft Label} \\
\text{Assignment (TESLA)} \\
\hline
\textbf{Input: } f : \text{teacher model}; \, \Theta : \text{teacher model's trajecto-} \\
\text{ries}; \, K: \text{number of iterations}; \, T: \text{number of matching} \\
\text{steps}; \, \beta: \text{learning rate for student model}; \, \alpha: \text{learning rate} \\
\text{for the synthetic images.} \\
\begin{aligned}
& \textbf{for } \text{iter} = 1 \dots K \textbf{do} \\
& \quad \text{Sample } \theta_t^* \text{ and } \theta_{t+M}^* \in \Theta, \text{ set } G = 0, \, \hat{\theta}_t = \theta_t^* \\
& \quad \text{Initialize } \tilde{Y} = f(\theta_{t+M}^*; \tilde{X}) \ \{\text{Soft Label Assignment}\} \\
& \quad \textbf{for } i = 1, \dots, T \textbf{ do} \\
& \quad\quad \text{Compute } g_i = \nabla_{\theta}\ell(\hat{\theta}_{t+i}; \tilde{X}_i) \\
& \quad\quad \text{Update } \hat{\theta}_{t+i} = \hat{\theta}_{t+i-1} - \beta g_i; \quad G = G + g_i \\
& \quad \textbf{end for} \\
& \quad \textbf{for } i = 1, \dots, T \textbf{ do} \\
& \quad\quad \text{Compute } g_i = \nabla_{\theta}\ell(\hat{\theta}_{t+i}; \tilde{X}_i) \\
& \quad\quad \text{Compute } \frac{\partial \|\hat{\theta}_{t+T} - \theta_{t+M}^*\|_2^2}{\partial \tilde{X}_i} \text{ based on } g_i \text{ and Equa-} \\
& \quad\quad \text{tion } (*) \\
& \quad\quad \frac{\partial \|\hat{\theta}_{t+T} - \theta_{t+M}^*\|_2^2}{\partial x_{\pi_{i,b}}} += \frac{\partial \|\hat{\theta}_{t+T} - \theta_{t+M}^*\|_2^2}{\partial \tilde{X}_{i,b}} \text{ for all } b \\
& \quad \textbf{end for} \\
& \quad \text{Update } x_j \text{ using } \frac{\partial \|\hat{\theta}_{t+T} - \theta_{t+M}^*\|_2^2}{\partial x_j} \text{ for all sampled } j \\
& \textbf{end for}
\end{aligned} \\
\hline
\end{array}$$

# 总结

在使用 Soft Label 技巧下的 TESLA 方法稳定地略强于 MTT 的表现，并且显存占据极大程度下降。

实际上 TESLA 方法存在极其轻微的代价，就是极少量的重复计算。这很容易理解，因为我们进行两次循环的过程中局部 mini-batch 的计算实际上被重复了。此处有一个优化，也许是缓存这些重复计算结果。但是无论如何，这种轻微的重复计算却带来了巨大的显存降低便利，这绝对是我们愿意做的。

TESLA 还揭露了 MTT 的一个缺陷，那就是架构迁移性问题。这对于数据集蒸馏方法而言几乎是顽疾。虽然我们提到 MTT 表现出来一定架构迁移性，但是当把对于 ConvNet 蒸馏的数据集迁移到 ResNet-18 上，模型识别准确率会暴跌。这是待解决的问题。

下一章我们仍围绕 MTT 展开，我们来讲讲 DATM，他们探讨了 MTT 在不同段的监督难度。